In [ ]:
!pip install 'litellm'==1.44.9
!pip install 'litellm[proxy]'==1.44.9
!pip install langfuse==2.52.2

In [1]:
import os
from google.colab import userdata
from pprint import pprint
from IPython.display import display, Markdown,HTML
from IPython.core.display import json
from litellm import completion,acompletion
import litellm
import openai

In [2]:
os.environ['Groq_API_Key'] = userdata.get('Groq_API_Key')
os.environ['COHERE_API_KEY'] = userdata.get('COHERE_API_KEY')

os.environ['LANGFUSE_SECRET_KEY'] = userdata.get('LANGFUSE_SECRET_KEY')
os.environ['LANGFUSE_PUBLIC_KEY'] = userdata.get('LANGFUSE_PUBLIC_KEY')
os.environ['LANGFUSE_HOST'] = "https://cloud.langfuse.com"

## SDK Logs

In [3]:
logs_dir = './LLM_Logs'
os.makedirs(logs_dir, exist_ok=True)

# from LiteLLM Docs
def success_callback(
    kwargs,                 # kwargs to completion
    completion_response,    # response from completion
    start_time, end_time    # start/end time
):
    with open(f"{logs_dir}/success_Logs.jsonl", "a") as f:
        f.write(json.dumps({
            "kwargs": kwargs,
            "completion_response": completion_response,
            "start_time": start_time,
            "end_time": end_time
        },ensure_ascii=False,default=str) + "\n")


def failure_callback(
  kwargs,                 # kwargs to completion
  completion_response,    # response from completion
  start_time, end_time    # start/end time
):
  with open(f"{logs_dir}/failure_Logs.jsonl", "a") as f:
        f.write(json.dumps({
            "kwargs": kwargs,
            "completion_response": completion_response,
            "start_time": start_time,
            "end_time": end_time
        },ensure_ascii=False,default=str) + "\n")

# Assign the custom callback function
litellm.success_callback = [success_callback]
litellm.failure_callback = [failure_callback]

## Groq LLAMA 3.1

In [4]:
messages=[
       {"role": "user",
        "content": "لماذا تبدو السماء زرقاء بانهار؟"}
   ]

response = completion(
    model="groq/llama-3.1-8b-instant",
    messages= messages,
    api_key=os.environ['Groq_API_Key'],
    #stream=True,
)


In [5]:
pprint(response['choices'][0]['message']['content'])

('يبدو اللون الأزرق في السماء بانهار بسبب انتشار الأشعة الفوتوفوتونية التي '
 'يبعثها الشمس أثناء سفرها عبر الغلاف الجوي للأرض. الأشعة الفوتوفوتونية هي نوع '
 'من أنواع الإشعاع electromagnet الذي يحتوي على مختلف الطوابع الطيفية.\n'
 '\n'
 'عندما يمر الأشعة الفوتوفوتونية من الشمس عبر الغلاف الجوي للأرض، فإنها تتفاعل '
 'مع الغازات والجزيئات في الهواء. هذه التفاعلات تجعل الأشعة الفوتوفوتونية '
 'تنتشر في جميع أنحاء الطيف electroMagnetic، ولكنها تتحول إلى الطيف المرئي '
 '(الضوء) عند الطوابع الطيفية من 380-780 نانو متر.\n'
 '\n'
 'الغلاف الجوي للأرض مكون بشكل أساسي من النيتروجين، والأوكسجين، والهواء. عند '
 'التفاعل مع هذه الغازات، يخضع الأشعة الفوتوفوتونية لتأثير الانكسار، مما '
 'يجعلها تنتشر بشكل غير منتظم وتنتقل إلى جميع الاتجاهات.\n'
 '\n'
 'اللون الأزرق ينتج بسبب تفاعل الأشعة الفوتوفوتونية مع الغاز الأكسجين في أعلى '
 'الغلاف الجوي. في هذا المنطقة من الغلاف الجوي، تكون الشريحة العريضة من الأشعة '
 'الفوتوفوتونية من الطيف الأزرق، لأنها تتسبب في انكسار الأشعة الفوتوفوتونية في '
 'الزا

## Cohere

In [6]:
response_cohere = completion(
    model="cohere_chat/command-a-03-2025",
    messages = messages,
    api_key=os.environ['COHERE_API_KEY'],
    max_tokens= 200,
    temperature=0.5,
    #stream=True,
)

In [7]:
response_cohere.model

'command-a-03-2025'

In [8]:
display(Markdown(response_cohere['choices'][0]['message']['content']))

تبدو السماء زرقاء خلال النهار بسبب ظاهرة تسمى "تبعثر رايلي" (Rayleigh scattering). هذه الظاهرة تحدث عندما تتفاعل أشعة الشمس مع جزيئات الغلاف الجوي للأرض. إليك شرح مبسط:

1. **ضوء الشمس**: يتكون ضوء الشمس من طيف كامل من الألوان (قوس قزح)، وكل لون له طول موجي مختلف. الضوء الأزرق له طول موجي أقصر مقارنة بالضوء الأحمر.

2. **تبعثر رايلي**: عندما تدخل أشعة الشمس الغلاف الجوي للأرض، تتفاعل مع جزيئات الهواء (مثل النيتروجين والأكسجين). الجزيئات الصغيرة تشتت الضوء ذي الأطوال الموجية الأقصر (مثل الأزرق والبنفسجي) بشكل أكبر من الضوء ذي الأطوال الموجية الأطول (مثل الأحمر والأص

# Proxy

In [37]:
!fuser -k 7000/tcp

7000/tcp:            16893


In [57]:
###Check any litellm process###
#!pgrep-f1 litellm

###kill any litellm process
!pkill -f litellm

In [10]:
%%writefile llm.yaml
model_list:
  - model_name: "groq-gemma2-9b"
    litellm_params:
      model: groq/llama-3.1-8b-instant
      api_key: "os.environ/Groq_API_Key"

  - model_name: "cohere-command-a"
    litellm_params:
      model: cohere_chat/command-a-03-2025
      api_key: "os.environ/COHERE_API_KEY"

Writing llm.yaml


In [14]:
#to run any command in the background in colab, put 'nohup' in the begining and '&' in the end
!nohup litellm --port 7000 --config llm.yaml &
!sleep 10 && tail nohup.out

nohup: appending output to 'nohup.out'
Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new


LiteLLM: Proxy initialized with Config, Set models:
    groq-gemma2-9b
    cohere-command-a
INFO:     Started server process [8842]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:7000 (Press CTRL+C to quit)


In [6]:
messages=[
       {"role": "user",
        "content": "لماذا تبدو السماء زرقاء بانهار؟"}
   ]

In [9]:
client = openai.OpenAI(
    api_key="anything",
    base_url="http://0.0.0.0:7000"
)

# request sent to cohere v2 model
response_openai = client.chat.completions.create(model="cohere-command-a",
                                          messages = messages)

In [17]:
display(Markdown(response_openai.choices[0].message.content))

تبدو السماء زرقاء خلال النهار بسبب ظاهرة تسمى **تبعثر رايلي** (Rayleigh scattering). إليك شرح مبسط لهذه الظاهرة:

1. **ضوء الشمس**: يتكون ضوء الشمس من طيف كامل من الألوان (الأحمر، البرتقالي، الأصفر، الأخضر، الأزرق، النيلي، والبنفسجي)، والتي تندمج معًا لتشكل الضوء الأبيض.

2. **تبعثر الضوء**: عندما يدخل ضوء الشمس الغلاف الجوي للأرض، يصطدم بجزيئات الهواء والغبار والجزيئات الأخرى في الغلاف الجوي. هذه الجزيئات صغيرة جدًا مقارنة بطول موجة الضوء.

3. **تبعثر رايلي**: وفقًا لظاهرة تبعثر رايلي، يتم تبعثر (تشتت) الضوء ذو الأطوال الموجية القصيرة (مثل الأزرق والبنفسجي) بشكل أكبر من الضوء ذو الأطوال الموجية الطويلة (مثل الأحمر والأصفر). وذلك لأن قدرة التبعثر تتناسب عكسياً مع الطول الموجي للضوء مرفوعاً إلى القوة الرابعة.

4. **اللون الأزرق**: على الرغم من أن الضوء البنفسجي له طول موجي أقصر من الأزرق، إلا أن العين البشرية أكثر حساسية للون الأزرق، كما أن جزءًا من الضوء البنفسجي يتم امتصاصه أو تبعثره في الاتجاهات الأخرى. لذلك، يهيمن اللون الأزرق على ما نراه.

5. **السماء الزرقاء**: نتيجة لذلك، عندما ننظر إلى السماء خلال النهار، نرى الضوء الأزرق المتبعثر في جميع الاتجاهات، مما يجعل السماء تبدو زرقاء.

باختصار، السماء تبدو زرقاء خلال النهار بسبب تبعثر الضوء الأزرق في الغلاف الجوي للأرض.

# Load Balancer

In [2]:
%%writefile llm_lb.yaml
model_list:
  - model_name: "global-LLM"
    litellm_params:
      model: groq/llama-3.1-8b-instant
      api_key: "os.environ/Groq_API_Key"
      rpm: 20

  - model_name: "global-LLM"
    litellm_params:
      model: cohere_chat/command-a-03-2025
      api_key: "os.environ/COHERE_API_KEY"
      rpm: 10
router_settings:
 routing_strategy: simple-shuffle

Writing llm_lb.yaml


In [10]:
!nohup litellm --port 7000 --config llm_lb.yaml &
!sleep 10 && tail nohup.out

nohup: appending output to 'nohup.out'
Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new


LiteLLM: Proxy initialized with Config, Set models:
    global-LLM
    global-LLM
INFO:     Started server process [8675]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:7000 (Press CTRL+C to quit)


In [11]:
import openai

client = openai.OpenAI(
    api_key="anything",
    base_url="http://0.0.0.0:7000"
)

# request sent to cohere v2 model
response_openai = client.chat.completions.create(model="global-LLM",
                                          messages = messages)

In [12]:
response_openai.model

'groq/llama-3.1-8b-instant'

In [13]:
display(Markdown(response_openai.choices[0].message.content))

السماء تبدو زرقاء بانهار لأن الضوء الذي يصلنا من الشمس أو من مصادر أخرى يمر عبر طبقات من الغاز في الهواء، والتي تُسمى بالتريبتات أو الطرق. في تلك النقاط، يتأثر الضوء بفعل تأثير دوبلر والتأثير العاكسي للغيض. 

ينتج هذا التأثير عن اختلاف السرعة في التيار الكهربائي بين السحاب والهواء السفلي. يتم تكسير الإشعاعات الضوئية إلى طروحات مختلفة. بعضهم هو الضوء الأزرق والخضراء، الذي يتأثر بسرعة أكبر في تلك المواد، في حين أن الضوء الأحمر والأصفر، أقل تأثيرا.

وإذا كان هناك نزوح (انهار) في تلك المواد، سوف نران أزرق أكثر في المنحدرات العلوية لذلك، وتظهر السماء زرقاء.

# FallBacks

In [58]:
%%writefile llm_fallbacks.yaml
model_list:
  - model_name: "groq-gemma2-9b"
    litellm_params:
      model: groq/llama-3.1-8b-instant
      api_key: "os.environ/Groq_API_Key"
      rpm: 5

  - model_name: "cohere-command-a"
    litellm_params:
      model: cohere_chat/command-a-03-2025
      api_key: "os.environ/COHERE_API_KEY"
      rpm: 6

router_settings:
  enable_pre_call_check: true
  routing_strategy: simple-shuffle

litellm_settings:
  num_retries: 3
  success_callback : ["langfuse"]
  failure_callback : ["langfuse"]
  drop_params: true
  redact_user_api_key_info: true
  request_timeout: 10
  fallbacks: [{"cohere-command-a": ["groq-gemma2-9b"]}]
  allowed_fails: 3
  cooldown_time: 10

Overwriting llm_fallbacks.yaml


In [59]:
!nohup litellm --port 4000 --config llm_fallbacks.yaml &
!sleep 10 && tail nohup.out

nohup: appending output to 'nohup.out'
INFO:     127.0.0.1:57226 - "POST /chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:57226 - "POST /chat/completions HTTP/1.1" 200 OK
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [19442]
INFO:     Started server process [21381]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:4000 (Press CTRL+C to quit)


In [54]:
messages=[
       {"role": "user",
        "content": "لماذا تبدو السماء زرقاء بانهار؟"}
   ]

In [55]:
client = openai.OpenAI(
    api_key="anything",
    base_url="http://0.0.0.0:4000"
)

# request sent to cohere v2 model
response_openai = client.chat.completions.create(model="cohere-command-a",
                                          messages = messages)

# Observability

In [4]:
%%writefile llm_langfuse.yaml

model_list:
  - model_name: "groq-gemma2-9b"
    litellm_params:
      model: groq/llama-3.1-8b-instant
      api_key: "os.environ/Groq_API_Key"
      rpm: 5
  - model_name: "cohere-command-a"
    litellm_params:
      model: cohere_chat/command-a-03-2025
      api_key: "os.environ/COHERE_API_KEY"
      rpm: 6

router_settings:
  enable_pre_call_check: true
  routing_strategy: simple-shuffle

litellm_settings:
  success_callback : ["langfuse"]
  failure_callback : ["langfuse"]
  drop_params: true
  redact_user_api_key_info: true
  num_retries: 3
  request_timeout: 10
  fallbacks: [{"cohere-command-a": ["groq-gemma2-9b"]}]
  allowed_fails: 3
  cooldown_time: 10

Writing llm_langfuse.yaml


In [5]:
!nohup litellm --port 7000 --config llm_langfuse.yaml &
!sleep 10 && tail nohup.out

nohup: appending output to 'nohup.out'
INFO:     Started server process [4258]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:7000 (Press CTRL+C to quit)


In [6]:
client = openai.OpenAI(
    api_key="anything",
    base_url="http://0.0.0.0:7000"
)

messages=[
       {"role": "user",
        "content": "لماذا تبدو السماء زرقاء بانهار؟"}
   ]

# request sent to cohere v2 model
response_openai = client.chat.completions.create(model="cohere-command-a",
                                          messages = messages)

In [7]:
display(Markdown(response_openai.choices[0].message.content))

السؤال المثير! يُعزى اللون الأزرق في السماء أثناء الانهيار إلى شيء يسمى "تفاوت طول موجي". عندما يتضح الشمس وتتسلّق أعلى السماء، تتوزع الضوء الذي يتوهج منها بجميع أنواع الألوان. ومع ذلك، لا ن看见 جميع هذه الألوان في السماء لأن بعضها منخفض الطول الموجي، مما يعني أنه منخفض التردد، بينما البعض الآخر مرتفع الطول الموجي (أو عالي التردد).

أولئك الطويلة الموجة كالبرق، والتي تكون على جزء أكبر مما نراه. وعندما تُطابق الطويلة الموجة (أو الضوء الأبيض) مع سطح الأرض، تتم حصولنا على الألوان. حيث يتشتت طليعة البرق (أو الضوء الأبيض) إلى جميع أطياف الألوان التي نراها في الأصباغ الأزرق التي نراها في السماء.

وإذا استمر الانهيار لمدة طويلة، تبدو الألوان الأخرى على مدى الطيف الأصغر (البرق الأبيض) في أبعاد منخفضة، ويُدعى هذا بالضوء المتأخر. وتتلاّش الألوان المتغيرة بمرور الوقت، وتتم حصولنا على لون أزرق قوي وجميل يراه البعض.

# Langchain + LiteLLM

In [ ]:
!pip install langchain-openai langchain langchain-community

In [11]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader([
                           "https://openai.com/index/learning-to-reason-with-llms/",
                           "https://huggingface.co/blog/dpo-trl",
                           "https://www.deeplearning.ai/the-batch/how-agents-can-improve-llm-performance/"
])
docs = loader.load()

In [20]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
#from langchain_core.chat_models import ChatOpenAI

In [24]:
# No API key needed when using proxy
llm = ChatOpenAI(
    openai_api_base="http://0.0.0.0:7000",  # Your proxy URL
    model="cohere-command-a",
    api_key="anything",
    temperature=0.1,
)

In [25]:
map_prompt = ChatPromptTemplate.from_messages(
    [("system", "Write a concise summary of the following:\\n\\n{context}")]
)

map_chain = map_prompt | llm | StrOutputParser()

In [26]:
# Invoke chain
result = map_chain.invoke({"context": docs})

In [27]:
result

'**Summary:**\n\nThe provided documents discuss advancements in fine-tuning large language models (LLMs) and improving their performance through agent workflows. \n\n1. **Fine-tuning Llama 2 with Direct Preference Optimization (DPO):**  \n   - DPO simplifies the process of aligning LLMs with human preferences by bypassing the need for reward modeling and RL-based optimization.  \n   - It directly optimizes the language model on preference data using a binary cross-entropy loss, reducing complexity compared to traditional RLHF methods.  \n   - The TRL library supports DPO training, enabling fine-tuning of models like Llama v2 on datasets like stack-exchange preference pairs.  \n   - Key metrics during training include rewards for chosen/rejected responses and accuracies, ensuring alignment with human preferences.  \n\n2. **Improving GPT-4 and GPT-3.5 Performance with AI Agent Strategies:**  \n   - Agent workflows significantly enhance LLM performance by enabling iterative processes, suc

In [29]:
llm.model

'cohere-command-a'